<a href="https://colab.research.google.com/github/41340202s-jpg/ProgramingLanguage/blob/main/HW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_Part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-generativeai

In [ ]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

In [ ]:
from google.colab import userdata
from google import genai

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash'

In [ ]:
response = client.models.generate_content(
    model = MODEL_ID, contents="Explain how AI works in a few words"
)
print(response.text)

AI learns patterns from data to perform tasks or make decisions.


In [ ]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1XX4fBlW1StXRdhggy-asWHJl67qN3OBGs6I4FVYO6Eo/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"

REQUIRED_COLUMNS = ["日期", "科目", "作業成績"]

_auth_done = False
_gc = None
_ws = None

In [ ]:
# --- 主要功能區塊 ---
def get_user_grades():
    """
    透過終端機輸入學生成績，直到使用者輸入 'q' 結束。
    """
    print("--- 準備輸入成績。輸入 'q' 來停止。---")
    grades = []
    while True:
        subject = input("請輸入科目（或輸入 'q' 停止）：")
        if subject.lower() == 'q':
            break

        grade = input(f"請輸入 {subject} 的成績：")
        try:
            grade = int(grade)
        except ValueError:
            print("成績必須是數字。請重新輸入。")
            continue

        today = datetime.now().strftime('%Y-%m-%d')
        grades.append([today, subject, grade])
        print(f"已記錄：日期: {today}, 科目: {subject}, 成績: {grade}\n")

    return grades

In [59]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [ ]:
new_grades = get_user_grades()

--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：國文
請輸入 國文 的成績：88
已記錄：日期: 2026-03-26, 科目: 國文, 成績: 88

請輸入科目（或輸入 'q' 停止）：英文
請輸入 英文 的成績：77
已記錄：日期: 2026-03-26, 科目: 英文, 成績: 77

請輸入科目（或輸入 'q' 停止）：數學
請輸入 數學 的成績：66
已記錄：日期: 2026-03-26, 科目: 數學, 成績: 66

請輸入科目（或輸入 'q' 停止）：q


In [ ]:
new_grades

[['2026-03-26', '國文', 88], ['2026-03-26', '英文', 77], ['2026-03-26', '數學', 66]]

In [60]:
summary_result = get_ai_summary(new_grades)
print(summary_result)


--- 正在呼叫 AI 模型生成摘要... ---
好的，這是根據您提供的成績列表所做的總結與常見迷思整理，全程不進行任何評分判斷。

---

### 成績摘要

這份成績列表記錄了學生在 **2026年3月26日** 於以下三個科目的表現：

*   **國文：88分**
*   **英文：77分**
*   **數學：66分**

整體而言，這三科的成績分布在**66分至88分**之間。

---

### 成績與學習的常見迷思整理

以下是一些關於成績和學習的常見迷思，旨在提供更全面的視角，而非針對特定分數進行評判：

1.  **迷思一：成績高低完全等同於能力高低或學習成敗。**
    *   **實情：** 成績是學生在特定時間、針對特定考試或作業表現的「一個」衡量指標。它受到多種因素影響，如考試當天的狀態、題目設計、評分標準、學習方法是否得當，以及個人情緒等。高分不代表完美無缺，低分也不代表完全沒有能力或努力。
2.  **迷思二：成績是唯一的學習評量標準，或決定未來成就的唯一因素。**
    *   **實情：** 雖然成績在學術和升學上扮演重要角色，但它無法完全涵蓋一個人的學習廣度、深度、批判性思考能力、解決問題的能力、創造力、團隊合作精神、抗壓性、求知慾等非量化特質。這些能力同樣對個人的成長和未來發展至關重要。
3.  **迷思三：單一科目成績可推斷學生對該領域的全部興趣或天賦。**
    *   **實情：** 學生在某個科目獲得較高的分數，可能反映了他們對該科目的掌握度較好，但不一定代表他們只對該科目感興趣或只擅長該領域。反之，一個科目的成績不理想，也可能是因為尚未找到適合的學習方法、教學方式不適合，或當時有其他因素干擾，而非表示對該領域毫無天賦或興趣。
4.  **迷思四：成績高就代表完全理解，成績低就代表沒有努力。**
    *   **實情：** 高分可能表示學生能有效掌握考試範圍內的知識點，但並不保證其能將知識融會貫通或應用於實際情境。低分也可能是因為學習策略不佳、未能理解考題意圖、或在學習過程中遇到瓶頸，而非單純缺乏努力。學習過程中的努力和方法同樣關鍵。

---


In [61]:
def main():
    """
    主程式流程：輸入成績 -> 獲取 AI 摘要 -> 寫入 Google Sheet。
    """
    try:
        # 1. Google Sheet 身份驗證
        auth.authenticate_user()

        creds, _ = default()
        gc = gspread.authorize(creds)

        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)





        print("--- Google Sheet 連線成功。---")

        # 2. 獲取使用者輸入的成績
        new_grades = get_user_grades()

        if not new_grades:
            print("沒有輸入任何成績，程式結束。")
            return

        # 3. 將新成績寫入 Google Sheet
        ws.append_rows(new_grades)
        print("\n--- 成績已成功寫入 Google Sheet。---")

        # 4. 獲取 AI 摘要並寫入 Google Sheet
        summary = get_ai_summary(new_grades)

        # 尋找第一行空白列
        next_row = len(ws.col_values(1)) + 1

        # 使用 update_cell() 方法逐一更新儲存格
        ws.update_cell(next_row, 1, datetime.now().strftime('%Y-%m-%d'))
        ws.update_cell(next_row, 2, 'AI 摘要')

        # 為了避免單元格內容過長，將摘要內容分成多行來寫入
        summary_lines = summary.split('\n')
        for i, line in enumerate(summary_lines):
            ws.update_cell(next_row + i, 3, line)

        print("\n--- AI 摘要已成功寫入 Google Sheet。---")
        print("以下是 AI 生成的摘要內容：")
        print("-" * 50)
        print(summary)
        print("-" * 50)

    except gspread.exceptions.APIError as e:
        print(f"Google Sheets API 錯誤：{e.response.text}")
        print("請確認：")
        print("1. 您的服務帳戶金鑰檔案正確且未過期。")
        print("2. 您已將服務帳戶的 Email 地址（在 JSON 檔案中）分享給 Google Sheet，並給予編輯權限。")
    except Exception as e:
        print(f"發生未預期的錯誤：{e}")

if __name__ == "__main__":
    main()

--- Google Sheet 連線成功。---
--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：國文
請輸入 國文 的成績：88
已記錄：日期: 2026-03-26, 科目: 國文, 成績: 88

請輸入科目（或輸入 'q' 停止）：英文
請輸入 英文 的成績：78
已記錄：日期: 2026-03-26, 科目: 英文, 成績: 78

請輸入科目（或輸入 'q' 停止）：數學
請輸入 數學 的成績：55
已記錄：日期: 2026-03-26, 科目: 數學, 成績: 55

請輸入科目（或輸入 'q' 停止）：q

--- 成績已成功寫入 Google Sheet。---

--- 正在呼叫 AI 模型生成摘要... ---

--- AI 摘要已成功寫入 Google Sheet。---
以下是 AI 生成的摘要內容：
--------------------------------------------------
好的，這是一份根據您提供的成績列表所做的簡單摘要與常見迷思整理，全程將不對成績進行任何評斷。

---

### **學生學業表現摘要**

這份成績列表顯示了學生在2026年3月26日三科的表現。

*   **國文：** 88分
*   **英文：** 78分
*   **數學：** 55分

成績分佈範圍從55分（數學）到88分（國文）。

---

### **關於學業成績的常見迷思整理**

在檢視學生成績時，人們常會有一些既定的觀念或迷思。以下整理幾項常見的迷思，旨在提供更全面的視角，而非對單一成績進行判斷：

1.  **迷思一：單一成績能完全定義學生的學習能力與潛力。**
    *   **事實：** 成績是學習過程中某個時間點的快照，反映了學生在特定測驗中對特定知識點的掌握程度。它無法完全涵蓋學生的學習熱情、解決問題的能力、批判性思維、創造力、課堂參與度、社交技能或在其他領域的潛力。

2.  **迷思二：分數高低是衡量努力程度的唯一標準。**
    *   **事實：** 學習成績受多種因素影響，包括學習方法是否得當、考試形式、題目設計、學生的理解能力、當天考試狀態、對科目的興趣程度，以及教師的教學方式等。單一分數的高低不應直接被解讀為努力程度的唯一指標。

3.  **迷思